In [1]:
!pip install torch torchvision torchaudio decord scikit-learn

  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 283.6 kB/s eta 0:00:0000:0100:57
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 595.4 kB/s eta 0:00:0000:0100:25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 483.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 372.9 kB/s eta 0:00:0000:0100:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 360.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 567.8 kB/s eta 0:00:0000:0100:20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 716.3 kB/s eta 0:00:0000:0100:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 521.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 704.1 kB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 530.1 kB/s eta 0:00:0000:0100:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18
from decord import VideoReader, cpu
import numpy as np
import os
from sklearn.model_selection import train_test_split

In [ ]:
# Specify dataset root with absolute path (adjust to your exact location)
DATA_ROOT = '/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data'  # Example; change if different

In [ ]:
# Class-specific directories (each contains cam1/ and cam2/ with MP4s)
VIOLENT_DIR = os.path.join(DATA_ROOT, 'violent')
NON_VIOLENT_DIR = os.path.join(DATA_ROOT, 'non-volent')  # Used 'non-volent' as per your query; change to 'non-violent' if typo

In [ ]:
# Quick check for directory existence
print(f"Current notebook working directory: {os.getcwd()}")
for dir_path in [DATA_ROOT, NON_VIOLENT_DIR, VIOLENT_DIR]:
    if os.path.exists(dir_path):
        print(f"{dir_path} exists.")
    else:
        print(f"WARNING: {dir_path} does NOT exist. Check your path!")

In [ ]:
# CSV files (for reference; not used in binary classification but can be loaded if needed)
VIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'violent-action-classes.csv')
NONVIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'nonviolent-action-classes.csv')
ACTION_OCCURRENCES_CSV = os.path.join(DATA_ROOT, 'action-class-occurrences.csv')

In [ ]:
# Class labels
CLASSES = ['non-volent', 'violent']  # Match your folder names

In [ ]:
# Function to recursively find MP4 files in a directory (case-insensitive extension, with debugging prints)
def find_mp4_files(directory):
    mp4_files = []
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist. Skipping.")
        return mp4_files
    for root, _, files in os.walk(directory):
        print(f"Searching in subfolder: {root}")
        for file in files:
            if file.lower().endswith('.mp4'):
                full_path = os.path.join(root, file)
                mp4_files.append(full_path)
                print(f"Found MP4: {full_path}")
    return mp4_files

# Collect MP4 paths for each class with debugging
non_violent_mp4s = find_mp4_files(NON_VIOLENT_DIR)
violent_mp4s = find_mp4_files(VIOLENT_DIR)

print(f"\nSummary:")
print(f"Found {len(non_violent_mp4s)} non-violent MP4s: {non_violent_mp4s[:5]}")  # First 5 for brevity
print(f"Found {len(violent_mp4s)} violent MP4s: {violent_mp4s[:5]}")

# Assign labels
non_violent_data = [(path, 0) for path in non_violent_mp4s]
violent_data = [(path, 1) for path in violent_mp4s]
all_data = non_violent_data + violent_data

# Check if data is empty
if not all_data:
    raise ValueError("No MP4 files found! Verify: 1) Absolute path in DATA_ROOT (Section 2). 2) Folder names (e.g., 'non-volent' exact match). 3) Run !ls {NON_VIOLENT_DIR}/cam1 in a cell to check files. 4) Ensure files end with .mp4 (case-insensitive).")

# Unpack and split (stratified by label for balance)
paths, labels = zip(*all_data)

# Split into train + temp (85% train+val, 15% test)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=42
)

# Split train_val into train and val (approx 70/15/15 overall)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.1765, stratify=train_val_labels, random_state=42
)

print(f"Train samples: {len(train_paths)}, Val samples: {len(val_paths)}, Test samples: {len(test_paths)}")

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, clip_len=16, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.clip_len = clip_len
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        vid_path = self.video_paths[idx]
        label = self.labels[idx]

        # Load video with Decord
        vr = VideoReader(vid_path, ctx=cpu(0))
        total_frames = len(vr)

        # Sample frames uniformly
        if total_frames >= self.clip_len:
            indices = np.linspace(0, total_frames - 1, self.clip_len, dtype=int)
        else:
            indices = np.tile(np.arange(total_frames), self.clip_len // total_frames + 1)[:self.clip_len]

        frames = vr.get_batch(indices).asnumpy()  # Shape: (clip_len, H, W, C)

        # Convert to tensor (C, T, H, W) and normalize
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).float() / 255.0
        if self.transform:
            frames = self.transform(frames)

        return frames, label

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # Standard input size for 3D ResNet
    transforms.Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])

# Create datasets using split paths
train_dataset = VideoDataset(train_paths, train_labels, clip_len=16, transform=transform)
val_dataset = VideoDataset(val_paths, val_labels, clip_len=16, transform=transform)
test_dataset = VideoDataset(test_paths, test_labels, clip_len=16, transform=transform)

# Data loaders
batch_size = 8  # Adjust based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pre-trained 3D ResNet and modify for 2 classes
model = r3d_18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # Output for violent/non-violent
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adjust LR as needed

In [ ]:
num_epochs = 10  # Adjust as needed

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {running_loss / len(train_loader):.4f}, Val Loss: {val_loss / len(val_loader):.4f}, Val Accuracy: {100 * correct / total:.2f}%')